# Vachan V2 — Per-Persona Tone Dial via Control Vectors

**What this proves (beyond the basic spike):** Instead of a *generic* Hinglish↔English axis, we build the tone vector **from a specific persona's own anchor phrases** — the Hinglish samples vs. their English-translated equivalents that Vachan already stores. The resulting dial is personalized: pushing it positive sounds *like that persona*, not just "generic casual Hinglish".

**Why this matters for Vachan:** The PFS gate already catches fidelity drift. When a persona consistently fails the cosine threshold (av_cosine < 0.68), this per-persona vector is Path-B — steer the model toward *their* voice without retraining.

**Prerequisite:** Run the [basic Hinglish spike](hinglish_control_vector_kaggle.ipynb) first and confirm it executes cleanly. This notebook assumes you've seen coeff ±2 shift the tone.

> Runtime: ~8–15 min on a Kaggle **T4** (set Accelerator → GPU T4 + Internet: On).

## 0. Setup

In [ ]:
!pip install -q repeng transformers accelerate bitsandbytes

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from repeng import ControlVector, ControlModel, DatasetEntry

MODEL = "NousResearch/Meta-Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token_id = tokenizer.eos_token_id

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map="auto")
model = ControlModel(model, list(range(-5, -18, -1)))
print("loaded:", MODEL)

## 1. Define the Persona

This is the only cell you change per-persona. Paste in:
- `PERSONA_NAME` — just a label for prints.
- `PERSONA_BIO` — the persona's system prompt (who they are, their tone, their context).
- `HINGLISH_ANCHORS` — 8–16 phrases **in the persona's actual voice**: casual, code-mixed, the way they'd WhatsApp a friend. Pull these from your persona card or generate them from the persona's voice profile.
- `ENGLISH_ANCHORS` — the **same meaning**, translated to formal English. These are the negative end of the axis.

The vector is trained on the *difference* between these two sets — so the closer these anchors are to the persona's real voice, the more personalized the dial.

In [ ]:
# ── CHANGE THIS CELL PER PERSONA ──────────────────────────────────────────────

PERSONA_NAME = "Priya"  # Label only — no effect on the vector

PERSONA_BIO = """
You are Priya, a 28-year-old product manager at a mid-size Delhi tech startup.
You speak Hinglish naturally — Hindi and English mixed mid-sentence.
You are warm, direct, slightly impatient, use WhatsApp-style shorthand.
You never say 'I hope this email finds you well'.
"""

# POSITIVE end of the dial — persona's natural casual voice
# 8–16 pairs works well; fewer = weaker vector, more = diminishing returns
HINGLISH_ANCHORS = [
    "haan bhai deployment ho gayi, check kar lo ek baar",
    "yaar kal meeting hai, prep kar lena thoda",
    "issue resolve ho gaya, ab chill karo",
    "client ko beta version bhej diya, response dekhte hain",
    "arey feature toh ready hai, bas testing pending hai",
    "sprint review kal hai, sab kuch push karo aaj",
    "database ka load test karte hain pehle, phir production",
    "okay so plan yeh hai — pehle MVP, baaki baad mein",
    "mujhe lag raha hai yeh approach better rahegi",
    "team se pooch lena, main busy hoon thodi der",
]

# NEGATIVE end of the dial — same content, formal English translation
ENGLISH_ANCHORS = [
    "The deployment has been completed. Please verify the environment.",
    "We have a meeting scheduled for tomorrow. Please prepare accordingly.",
    "The issue has been resolved. You may proceed.",
    "The beta build has been shared with the client. Awaiting their feedback.",
    "The feature is ready. Testing is the only remaining step.",
    "The sprint review is tomorrow. Please ensure all tasks are pushed today.",
    "We should conduct a load test on the database before the production push.",
    "The plan is as follows: deliver the MVP first, and address remaining items subsequently.",
    "I believe this approach would be more effective.",
    "Please consult the team. I am occupied at the moment.",
]

assert len(HINGLISH_ANCHORS) == len(ENGLISH_ANCHORS), "Each Hinglish anchor needs a paired English translation"
print(f"{PERSONA_NAME}: {len(HINGLISH_ANCHORS)} anchor pairs loaded")

## 2. Build the Contrastive Dataset

Same mechanics as the basic spike, but now the contrast is built from the **persona's own anchors** — not generic Hinglish/formal strings. The suffixes give the extractor many hidden-state positions to read from.

In [ ]:
GENERIC_SUFFIXES = [
    "", " I", " I think", " So", " Yeah", " Okay", " Sure",
    " Right", " Let me", " We", " The",
]

def make_entry(hinglish_anchor: str, english_anchor: str, suffix: str) -> DatasetEntry:
    def fmt(instruction: str) -> str:
        msgs = [
            {"role": "system", "content": PERSONA_BIO},
            {"role": "user",   "content": instruction},
        ]
        base = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        return base + suffix
    return DatasetEntry(
        positive=fmt(hinglish_anchor),
        negative=fmt(english_anchor),
    )

dataset = [
    make_entry(h, e, s)
    for h, e in zip(HINGLISH_ANCHORS, ENGLISH_ANCHORS)
    for s in GENERIC_SUFFIXES
]

print(f"Dataset: {len(dataset)} contrastive pairs ({len(HINGLISH_ANCHORS)} anchors × {len(GENERIC_SUFFIXES)} suffixes)")
print("--- sample positive (tail) ---")
print(dataset[0].positive[-180:])

## 3. Train the Per-Persona Vector

In [ ]:
model.reset()
persona_vector = ControlVector.train(model, tokenizer, dataset)

_layers = list(persona_vector.directions.keys())
print(f"{PERSONA_NAME} vector trained over {len(_layers)} layers")
print("direction shape:", persona_vector.directions[_layers[0]].shape)

## 4. Test the Dial — Same Prompt, Three Coefficients

Expected: `coeff +2` sounds like the persona's WhatsApp voice. `coeff -2` sounds like a formal English email. `coeff 0` is the base model.

If the shift is weak: add more anchor pairs or widen the layer band.  
If the output degenerates (repetition, nonsense): reduce coeff to 1.0–1.5.

In [ ]:
def generate(prompt: str, coeff: float) -> str:
    model.reset()
    if coeff != 0:
        model.set_control(persona_vector, coeff)
    msgs = [
        {"role": "system", "content": PERSONA_BIO},
        {"role": "user",   "content": prompt},
    ]
    ids = tokenizer.apply_chat_template(
        msgs, return_tensors="pt", add_generation_prompt=True
    ).to(model.device)
    out = model.generate(
        ids,
        max_new_tokens=90,
        do_sample=False,
        repetition_penalty=1.3,
        pad_token_id=tokenizer.eos_token_id,
    )
    model.reset()
    return tokenizer.decode(out[0, ids.shape[-1]:], skip_special_tokens=True).strip()


TEST_PROMPTS = [
    "Can you give me an update on the deployment?",
    "What's the plan for tomorrow's sprint review?",
    "The client is asking about the timeline.",
]

for prompt in TEST_PROMPTS:
    print(f"\n{'='*55}")
    print(f"PROMPT: {prompt}")
    print(f"{'='*55}")
    for c in [-2.0, 0.0, 2.0]:
        print(f"  coeff {c:+.1f} → {generate(prompt, c)}")

## 5. Save the Vector (Optional — for Reuse)

If the vector works well, save it so you don't retrain every time. Load it back and inject at generation time — no GPU needed for storage/loading.

In [ ]:
import json, numpy as np
from pathlib import Path

OUT_DIR = Path(f"vectors/{PERSONA_NAME.lower()}")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Save each layer's direction as a numpy array
for layer_idx, direction in persona_vector.directions.items():
    np.save(OUT_DIR / f"layer_{layer_idx}.npy", direction.cpu().float().numpy())

# Save metadata
meta = {
    "persona": PERSONA_NAME,
    "model": MODEL,
    "anchor_count": len(HINGLISH_ANCHORS),
    "layers": sorted(persona_vector.directions.keys()),
    "recommended_coeff_range": [-2.0, 2.0],
}
with open(OUT_DIR / "meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print(f"Vector saved → {OUT_DIR}/")
print(json.dumps(meta, indent=2))

## 6. What This Unlocks for Vachan

**Immediate:** You can run this for any persona card — swap out the anchors in Cell 1 and re-run. Each persona gets their own tone direction.

**Integration path (next sprint):**
1. **Fidelity Ring trigger** — when av_cosine drops below 0.68 for a persona, load their saved vector and re-run generation with coeff +1.5. This is Path-B.
2. **Coeff auto-tuning** — run the Fidelity Ring scorer on outputs at coeff 0.5, 1.0, 1.5, 2.0 and pick the coeff that maximizes av_cosine without degeneration.
3. **vLLM serving** — vectors are tiny (one float32 array per layer, ~4KB total). Load them at startup, apply via a custom vLLM sampler hook.

**Remaining question from this notebook:** Does the per-persona vector score *higher* on av_cosine than the generic Hinglish vector? That comparison is the next metric gate to add to CI.